# Methodology Analysis

This notebook summarizes the FP-Growth rule mining, the rule balance, and the basket-size analysis that informed the current demo.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

root = Path('..').resolve()
rules_path = root / 'data' / 'mined' / 'association_rules.csv'
analysis_path = root / 'results' / 'analysis' / 'strategy_summary.csv'

rules = pd.read_csv(rules_path)
analysis = pd.read_csv(analysis_path)

rules.head()

In [ ]:
rules['rule_type'].value_counts().plot(kind='bar', title='Rule count by type')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
for strategy, group in analysis.groupby('strategy'):
    group = group.sort_values('basket_size')
    plt.plot(group['basket_size'], group['top1_accuracy'], marker='o', label=f'{strategy} top-1')
    plt.plot(group['basket_size'], group['top3_accuracy'], marker='s', linestyle='--', label=f'{strategy} top-3')

plt.title('Cuisine prediction accuracy by basket size')
plt.xlabel('Basket size')
plt.ylabel('Accuracy')
plt.ylim(0, 1)
plt.grid(True, alpha=0.2)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
summary = rules.groupby('rule_type')[['support', 'confidence', 'lift', 'pmi']].agg(['mean', 'median', 'max'])
summary

## PMI tail check

This cell measures how low the support gets inside the top 1 percent of rules when ranked by PMI.

In [ ]:
pmi_rules = rules[rules['pmi'].notna()].copy()
pmi_cutoff = pmi_rules['pmi'].quantile(0.99)
top_pmi = pmi_rules[pmi_rules['pmi'] >= pmi_cutoff].copy()
top_pmi['abs_count'] = (top_pmi['support'] * 9340).round().astype(int)

print(f'PMI cutoff: {pmi_cutoff:.6f}')
print(f'Top 1% rule count: {len(top_pmi)}')
print(f'Lowest absolute count in top 1%: {top_pmi["abs_count"].min()}')

top_pmi.sort_values(['pmi', 'lift', 'confidence'], ascending=False)[['antecedent', 'consequent', 'abs_count', 'support', 'confidence', 'lift', 'pmi']].head(15)